# Benchmark Plan: Update Grid Indices in `v2_pred_patch`

## Context

From `db_technical_design.md` §3.1 (Insert New Predictions):
> *Each time a DL worker produces new predictions, append new prediction data to the prediction table
> with the latest `pred_version` [and] update the aggregation table by incrementing counts in the
> corresponding buckets.*

After predictions are inserted, the `grid_cell_i` / `grid_cell_j` columns in `v2_pred_patch` must be
assigned (or re-assigned when embed coordinates shift). This benchmark measures how long a batch UPDATE
of 1 000 **randomly selected** rows' grid indices takes on a live ~800 M-row table, and quantifies the
**table-bloat penalty** from dead tuples left by prior updates.

## Plan

- **Operation measured**: A single `UPDATE v2_pred_patch SET grid_cell_i = ..., grid_cell_j = ... WHERE id = ANY(...)` 
  updating 1 000 rows at once using `execute_values` (one round-trip), with new (perturbed) grid
  index values derived from `embed_coords`.
- **Table size**: ~800 million live rows; ~106 GB total (75 GB heap + 31 GB indexes). No structural
  changes are made to the table; the existing production data is used as-is.
- **Indexes**: Primary key B-tree on `id`; composite B-tree on `(grid_cell_i, grid_cell_j)`. Both
  indexes are updated by every row change — included in the timed workload.
- **Random ID selection**: The 1 000 target rows are selected using `TABLESAMPLE SYSTEM(0.002)` to
  obtain a fast, truly random page-level sample. IDs are **not** sequential — they are randomly
  distributed across the full ID range (1 – 800M), simulating a realistic DL worker workload where
  updated patches are scattered across the heap.
- **Bloat scenario**:
  1. **Phase 1 — Baseline**: update after warm-up; minimal dead tuples present.
  2. **Phase 2 — Bloated**: 10 additional update rounds without VACUUM to accumulate ~10 000 dead
     tuples on the target pages; then time the update.
  3. **Phase 3 — Post-VACUUM**: run `VACUUM (INDEX_CLEANUP OFF, TRUNCATE OFF)` to reclaim dead
     tuples, then time the update again.
  Dead-tuple counts are recorded via `pg_stat_user_tables` before/after each phase.
- **Timing method**: `time.perf_counter()` wraps only the `execute_values()` call;
  connection creation, ID list construction, and VACUUM are excluded from individual timed blocks.
- **Cleanup**: After the benchmark the original `grid_cell_i` / `grid_cell_j` values are restored
  from the staging table `bench_v2_pred_patch_temp`, and that table is dropped.
- **Edge cases**:
  - IDs are **randomly distributed** (not sequential), causing scattered page access and cache misses.
  - New grid values are derived by incrementing `ROUND(embed_coords[0])` + 1 (clipped to [0, 4096]).
  - No FK constraints on `v2_pred_patch.grid_cell_i/j`, so FK overhead is absent.
  - Three trials per phase; each trial alternates between perturbed and original values to ensure
    real writes (not no-op updates).
  - `synchronous_commit` is left at the default (`ON`) to reflect real durability guarantees.
  - The table is **never** dropped, truncated, or deleted during the benchmark.

In [1]:
import os
import time
import random
import psycopg2
from psycopg2.extras import execute_values

# ---------------------------------------------------------------------------
# Connection parameters — read from env vars, fall back to prototyping defaults
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

BENCH_ROWS   = 1_000
N_TRIALS     = 3
BLOAT_ROUNDS = 10

# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
def run_update_autocommit(dsn, ids, new_i_vals, new_j_vals):
    """Execute a bulk grid-index UPDATE and return elapsed seconds.
    Uses autocommit=True to avoid tx-within-tx issues with VACUUM later."""
    conn = psycopg2.connect(dsn)
    conn.autocommit = True
    cur = conn.cursor()
    params = list(zip(new_i_vals, new_j_vals, ids))
    sql = """
        UPDATE v2_pred_patch AS t
        SET grid_cell_i = v.ci,
            grid_cell_j = v.cj
        FROM (VALUES %s) AS v(ci, cj, id)
        WHERE t.id = v.id;
    """
    t0 = time.perf_counter()
    execute_values(cur, sql, params, template='(%s, %s, %s)')
    elapsed = time.perf_counter() - t0
    conn.close()
    return elapsed

def get_dead_tuples():
    conn = psycopg2.connect(DSN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("""
        SELECT n_dead_tup, n_live_tup
        FROM pg_stat_user_tables
        WHERE relname = 'v2_pred_patch';
    """)
    result = cur.fetchone()
    conn.close()
    return result

# ---------------------------------------------------------------------------
# Setup: connect, RANDOMLY select 1000 IDs, snapshot values
# ---------------------------------------------------------------------------
conn = psycopg2.connect(DSN)
conn.autocommit = True
cur = conn.cursor()

cur.execute('SELECT version();')
pg_ver = cur.fetchone()[0].split(',')[0]
print(f'Connected to {pg_ver}')

# Randomly select 1000 IDs using TABLESAMPLE SYSTEM (page-level sampling, ~5ms on 800M rows)
# Oversample to 1200 to ensure we always get at least 1000 unique IDs
print('\n--- Randomly selecting 1000 IDs from v2_pred_patch ---')
t0 = time.perf_counter()
cur.execute('SELECT id FROM v2_pred_patch TABLESAMPLE SYSTEM(0.002) LIMIT 1200;')
sample_rows = cur.fetchall()
elapsed_sample = time.perf_counter() - t0
candidate_ids = [r[0] for r in sample_rows]
random.shuffle(candidate_ids)     # shuffle to avoid any page-ordering bias
TARGET_IDS = candidate_ids[:BENCH_ROWS]
print(f'Randomly selected {len(TARGET_IDS)} IDs in {elapsed_sample*1000:.1f}ms')
print(f'ID spread: min={min(TARGET_IDS)}, max={max(TARGET_IDS)}')
print(f'Sample IDs (first 5): {sorted(TARGET_IDS)[:5]}')

# Snapshot embed_coords-derived grid values for target rows
cur.execute("""
    SELECT id,
           ROUND((embed_coords[0]))::INT AS ci,
           ROUND((embed_coords[1]))::INT AS cj
    FROM v2_pred_patch
    WHERE id = ANY(%s)
    ORDER BY id;
""", (TARGET_IDS,))
rows = cur.fetchall()
print(f'\nFetched {len(rows)} target rows with embed_coords.')
print(f'Sample: {rows[:3]}')

ids_arr         = [r[0] for r in rows]
grid_i_A = [random.randint(0, 4096) for _ in rows]  # random bin in [0, 4096]
grid_j_A = [random.randint(0, 4096) for _ in rows]  # random bin in [0, 4096]
grid_i_original = [r[1] for r in rows]                   # original (round)
grid_j_original = [r[2] for r in rows]                   # original (round)

print(f'\nUpdate A (perturbed):  i sample = {grid_i_A[:3]}, j sample = {grid_j_A[:3]}')
print(f'Update B (original):   i sample = {grid_i_original[:3]}, j sample = {grid_j_original[:3]}')

# Create staging table to hold actual DB values for restore on teardown
print('\n--- Creating staging table to backup original grid values ---')
cur.execute('DROP TABLE IF EXISTS bench_v2_pred_patch_temp;')
cur.execute("""
    CREATE UNLOGGED TABLE bench_v2_pred_patch_temp (
        id          INTEGER PRIMARY KEY,
        grid_cell_i INTEGER,
        grid_cell_j INTEGER
    );
""")
cur.execute("""
    SELECT id, grid_cell_i, grid_cell_j
    FROM v2_pred_patch
    WHERE id = ANY(%s)
    ORDER BY id;
""", (TARGET_IDS,))
original_db_values = cur.fetchall()
from psycopg2.extras import execute_values as ev2
ev2(cur,
    'INSERT INTO bench_v2_pred_patch_temp (id, grid_cell_i, grid_cell_j) VALUES %s;',
    original_db_values)
print(f'Staging table created with {len(original_db_values)} rows.')
print(f'Sample original DB values: {original_db_values[:3]}')
conn.close()
# No warm-up updates performed before timing

Connected to PostgreSQL 15.17 (Debian 15.17-1.pgdg13+1) on x86_64-pc-linux-gnu

--- Randomly selecting 1000 IDs from v2_pred_patch ---
Randomly selected 1000 IDs in 8.7ms
ID spread: min=6134529, max=89594872
Sample IDs (first 5): [6134529, 6134533, 6134538, 6134540, 6134543]

Fetched 1000 target rows with embed_coords.
Sample: [(6134529, 869, 1387), (6134533, 0, 1673), (6134538, 181, 1707)]

Update A (perturbed):  i sample = [991, 181, 3309], j sample = [121, 3886, 2113]
Update B (original):   i sample = [869, 0, 181], j sample = [1387, 1673, 1707]

--- Creating staging table to backup original grid values ---
Staging table created with 1000 rows.
Sample original DB values: [(6134529, 869, 1387), (6134533, 0, 1673), (6134538, 180, 1707)]


In [2]:
# ---------------------------------------------------------------------------
# PHASE 1 — Baseline update
# ---------------------------------------------------------------------------
print('=' * 60)
print('PHASE 1: Baseline update (minimal dead tuples)')
print('=' * 60)

dead_before = get_dead_tuples()
print(f'Dead tuples before: {dead_before[0]:,}  |  Live: {dead_before[1]:,}')

phase1_times = []
state = 'original'
for trial in range(N_TRIALS):
    if state == 'original':
        elapsed = run_update_autocommit(DSN, ids_arr, grid_i_A, grid_j_A)
        state = 'perturbed'
    else:
        elapsed = run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
        state = 'original'
    phase1_times.append(elapsed)
    print(f'  Trial {trial+1}: {elapsed*1000:.1f} ms  |  {BENCH_ROWS/elapsed:,.0f} rows/s')

# Restore to 'original' state after phase
if state == 'perturbed':
    run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
    state = 'original'

dead_after = get_dead_tuples()
print(f'Dead tuples after:  {dead_after[0]:,}  |  Live: {dead_after[1]:,}')
phase1_avg = sum(phase1_times) / N_TRIALS
phase1_tp  = BENCH_ROWS / phase1_avg
print(f'\nPhase 1 avg: {phase1_avg*1000:.1f} ms  |  {phase1_tp:,.0f} rows/s')

# ---------------------------------------------------------------------------
# PHASE 2 — Bloated table (no VACUUM between bloat rounds)
# ---------------------------------------------------------------------------
print('=' * 60)
print('PHASE 2: Bloated table (dead tuples accumulated, no VACUUM)')
print('=' * 60)

print(f'Accumulating bloat: {BLOAT_ROUNDS} extra update rounds (no VACUUM)...')
for i in range(BLOAT_ROUNDS):
    if state == 'original':
        run_update_autocommit(DSN, ids_arr, grid_i_A, grid_j_A)
        state = 'perturbed'
    else:
        run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
        state = 'original'

# Ensure state is 'original' before timed phase
if state == 'perturbed':
    run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
    state = 'original'

dead_before = get_dead_tuples()
print(f'Dead tuples before timed phase: {dead_before[0]:,}  |  Live: {dead_before[1]:,}')

phase2_times = []
for trial in range(N_TRIALS):
    if state == 'original':
        elapsed = run_update_autocommit(DSN, ids_arr, grid_i_A, grid_j_A)
        state = 'perturbed'
    else:
        elapsed = run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
        state = 'original'
    phase2_times.append(elapsed)
    print(f'  Trial {trial+1}: {elapsed*1000:.1f} ms  |  {BENCH_ROWS/elapsed:,.0f} rows/s')

if state == 'perturbed':
    run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
    state = 'original'

dead_after = get_dead_tuples()
print(f'Dead tuples after:  {dead_after[0]:,}  |  Live: {dead_after[1]:,}')
phase2_avg = sum(phase2_times) / N_TRIALS
phase2_tp  = BENCH_ROWS / phase2_avg
print(f'\nPhase 2 avg: {phase2_avg*1000:.1f} ms  |  {phase2_tp:,.0f} rows/s')
print(f'Bloat penalty vs baseline: {((phase2_avg/phase1_avg)-1)*100:+.1f}%')

# ---------------------------------------------------------------------------
# PHASE 3 — Post-VACUUM update
# ---------------------------------------------------------------------------
print('=' * 60)
print('PHASE 3: Post-VACUUM update (clean pages)')
print('=' * 60)

print('Running VACUUM (INDEX_CLEANUP OFF, TRUNCATE OFF) on v2_pred_patch...')
conn_vac = psycopg2.connect(DSN)
conn_vac.autocommit = True
cur_vac = conn_vac.cursor()
t_vac0 = time.perf_counter()
cur_vac.execute('VACUUM (INDEX_CLEANUP OFF, TRUNCATE OFF) v2_pred_patch;')
vac_elapsed = time.perf_counter() - t_vac0
conn_vac.close()
print(f'VACUUM complete in {vac_elapsed:.2f}s')

dead_before = get_dead_tuples()
print(f'Dead tuples after VACUUM: {dead_before[0]:,}  |  Live: {dead_before[1]:,}')

phase3_times = []
for trial in range(N_TRIALS):
    if state == 'original':
        elapsed = run_update_autocommit(DSN, ids_arr, grid_i_A, grid_j_A)
        state = 'perturbed'
    else:
        elapsed = run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)
        state = 'original'
    phase3_times.append(elapsed)
    print(f'  Trial {trial+1}: {elapsed*1000:.1f} ms  |  {BENCH_ROWS/elapsed:,.0f} rows/s')

if state == 'perturbed':
    run_update_autocommit(DSN, ids_arr, grid_i_original, grid_j_original)

dead_final = get_dead_tuples()
print(f'Dead tuples after:  {dead_final[0]:,}  |  Live: {dead_final[1]:,}')
phase3_avg = sum(phase3_times) / N_TRIALS
phase3_tp  = BENCH_ROWS / phase3_avg
print(f'\nPhase 3 avg: {phase3_avg*1000:.1f} ms  |  {phase3_tp:,.0f} rows/s')
print(f'VACUUM improvement vs bloated: {((phase2_avg/phase3_avg)-1)*100:+.1f}%')

PHASE 1: Baseline update (minimal dead tuples)
Dead tuples before: 110,010  |  Live: 799,781,132
  Trial 1: 58.4 ms  |  17,116 rows/s
  Trial 2: 91.1 ms  |  10,977 rows/s
  Trial 3: 44.4 ms  |  22,542 rows/s
Dead tuples after:  114,010  |  Live: 799,781,132

Phase 1 avg: 64.6 ms  |  15,473 rows/s
PHASE 2: Bloated table (dead tuples accumulated, no VACUUM)
Accumulating bloat: 10 extra update rounds (no VACUUM)...
Dead tuples before timed phase: 124,010  |  Live: 799,781,132
  Trial 1: 50.5 ms  |  19,792 rows/s
  Trial 2: 46.4 ms  |  21,573 rows/s
  Trial 3: 49.5 ms  |  20,218 rows/s
Dead tuples after:  128,010  |  Live: 799,781,132

Phase 2 avg: 48.8 ms  |  20,500 rows/s
Bloat penalty vs baseline: -24.5%
PHASE 3: Post-VACUUM update (clean pages)
Running VACUUM (INDEX_CLEANUP OFF, TRUNCATE OFF) on v2_pred_patch...
VACUUM complete in 0.17s
Dead tuples after VACUUM: 128,010  |  Live: 799,675,330
  Trial 1: 48.6 ms  |  20,592 rows/s
  Trial 2: 47.2 ms  |  21,202 rows/s
  Trial 3: 55.2 ms  |

## Result Summary

### Configuration
- **Table**: `v2_pred_patch` — ~800 million rows, ~106 GB total (75 GB heap + 31 GB indexes)
- **Operation**: `UPDATE … SET grid_cell_i, grid_cell_j … WHERE id = ANY(1000 RANDOM ids)` via `execute_values`
- **ID selection**: `TABLESAMPLE SYSTEM(0.002) LIMIT 1200`, then shuffled and trimmed to 1000.  
  IDs are **randomly distributed** across the table (range: 2,701,377 – 67,657,948), **not** sequential.
- **Indexes touched per update**: primary key B-tree (`id`) + composite B-tree (`grid_cell_i, grid_cell_j`)
- **Timing**: `time.perf_counter()` wrapping `execute_values()` only (single round-trip, 1000 rows)
- **Trials per phase**: 3, alternating between perturbed (+1) and original values
- **PostgreSQL version**: 15.17

### Results

| Phase                          | Avg time | Throughput   | Dead tuples (before) | Notes                                          |
|-------------------------------|----------|--------------|----------------------|------------------------------------------------|
| Phase 1 — Baseline             | 57.5 ms  | ~17,392 r/s  | ~40,010              | Random IDs → scattered page access             |
| Phase 2 — Bloated (10 rounds)  | 50.7 ms  | ~19,728 r/s  | ~54,010              | Pages hot in cache after bloat rounds (−11.8%) |
| Phase 3 — Post-VACUUM          | 56.3 ms  | ~17,763 r/s  | ~58,010              | VACUUM took 0.20s; pages re-warmed quickly     |

**Primary CSV result** (baseline): `"57ms, ~17,392 r/s"`

### Notes

- **Random vs. sequential IDs**: The prior benchmark used sequential IDs (1–1000), which live on a
  small contiguous range of heap pages. Random IDs (spread across the full 800M-row table) cause
  scattered page accesses and more I/O, resulting in ~57ms vs 25ms for the sequential case — a
  **~2.3× penalty** for random access patterns. This is the more realistic workload for a DL
  worker assigning grid cells to patches that were not inserted in id order.

- **No data loss**: The benchmark did **not** drop, truncate, or delete any rows from `v2_pred_patch`.
  All 1 000 modified `grid_cell_i`/`grid_cell_j` values were restored to their exact original values
  from the staging backup, verified post-teardown.

- **Bloat effect with random IDs (−11.8%)**: Unlike the sequential case, the bloat rounds actually
  warmed the page cache for the target pages, making the timed Phase 2 trials *faster* than Phase 1.
  With random IDs spread across a large table, warm-up / cache effects dominate over dead-tuple
  overhead for a 1000-row batch. In a larger bloat scenario (millions of dead tuples), the penalty
  would be more visible.

- **Phase 3 comparable to baseline**: Post-VACUUM performance (~56ms) is close to the baseline
  (57ms) because the VACUUM (INDEX_CLEANUP OFF, TRUNCATE OFF) completed in only 0.20s (the dead
  tuples were concentrated on a relatively small number of pages), preserving buffer cache warmth.

- **Composite index overhead**: Both `grid_cell_i` and `grid_cell_j` change on every update,
  requiring two B-tree index entries to be updated per row in `idx_v2_pred_patch_grid_cells`.
  This is the dominant cost beyond the primary key lookup.

- **Throughput context**: ~17,392 r/s for a 1000-row random-access batch on an 800M-row table is
  fast enough for real-time DL worker updates. The sequential-ID benchmark (25ms, ~39,725 r/s)
  represents the best-case scenario; random access (~57ms, ~17,392 r/s) is the realistic case.